# 05 - Trained Match Model and 2026 Group Predictions

Goal: train the first real match-outcome model using historical international results, then predict the 2026 World Cup group-stage fixtures.

This notebook is a real ML step because it has:

- `X`: feature differences between Team A and Team B
- `y`: historical result label (`home_win`, `draw`, `away_win`)
- train/test split
- model evaluation
- prediction on future fixtures

Important limitation: we are using 2026 squad-strength features to explain recent historical matches. That is not historically perfect. A stronger version would use FIFA ranking/Elo/player features as they existed on each match date. But this is a valid first trained baseline and a very good learning step.

In [1]:
import sys
from pathlib import Path

import pandas as pd

SRC_DIR = Path('../src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from trained_match_model import (
    RAW_DIR,
    PROCESSED_DIR,
    NAME_TO_CODE,
    load_inputs,
    add_missing_team_placeholders,
    prepare_results,
    build_training_data,
    train_models,
    predict_fixtures,
    build_group_tables,
)

print('Notebook 05 imports loaded')

Notebook 05 imports loaded


## Step 1 - Load inputs

We now have three key files:

- `final_team_features.csv`: team strength features from Notebook 03
- `results.csv`: historical international match results
- `wc_2026_group_fixtures.csv`: group-stage fixtures copied from the schedule you pasted

Always print schemas before modeling. If the columns are not what we expect, training results become meaningless.

In [2]:
teams, results, fixtures = load_inputs()

display(teams.head())
display(results.head())
display(fixtures.head())

teams: (45, 26)
['nation_code', 'nation_name', 'selection_method', 'squad_players_used', 'estimated_players_used', 'verified_supplemental_used', 'avg_selection_score', 'top6_goal_form_sum', 'top6_goal_form_mean', 'top10_chance_form_sum', 'top10_chance_form_mean', 'attacker_goal_form_mean', 'midfielder_chance_form_mean', 'defender_actions_per90_mean', 'keeper_save_pct_best', 'squad_weighted_minutes_mean', 'attack_score', 'creation_score', 'defense_score', 'keeper_score', 'depth_score', 'data_confidence_score', 'coverage_pct', 'has_announced_squad_file', 'raw_power_score', 'power_score']
results: (49287, 9)
['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
fixtures: (72, 7)
['group', 'match_number', 'date', 'time_utc_offset', 'team_a', 'team_b', 'venue']


,nation_code,nation_name,selection_method,squad_players_used,estimated_players_used,verified_supplemental_used,avg_selection_score,top6_goal_form_sum,top6_goal_form_mean,top10_chance_form_sum,...,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,coverage_pct,has_announced_squad_file,raw_power_score,power_score
0,ENG,England,official_squad,26,1,0,74.863565,3.281209,0.546868,1.454975,...,95.000000,96.777778,78.444444,86.666667,88.888889,98.653846,1.0,True,90.853333,90.547577
1,GER,Germany,official_squad,25,1,0,76.807714,2.591882,0.431980,1.338744,...,92.444444,94.444444,71.555556,84.444444,93.333333,98.600000,1.0,True,88.313333,88.004237
2,FRA,France,official_squad,26,1,0,77.163462,3.649274,0.608212,1.276214,...,96.555556,88.888889,45.555556,93.333333,95.555556,98.653846,1.0,True,84.972222,84.686258
3,ESP,Spain,official_squad,26,2,0,75.382369,2.518759,0.419793,0.963443,...,81.888889,74.555556,91.555556,95.555556,91.111111,97.307692,1.0,True,84.357778,83.789985
4,ARG,Argentina,official_squad_capped_top26,26,4,1,79.975258,2.476049,0.412675,1.330950,...,85.222222,88.000000,91.333333,18.888889,97.777778,94.615385,1.0,True,80.312222,79.231096


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


,group,match_number,date,time_utc_offset,team_a,team_b,venue
0,A,1,2026-06-11,1:00 p.m. UTC-6,Mexico,South Africa,"Estadio Azteca, Mexico City"
1,A,2,2026-06-11,8:00 p.m. UTC-6,South Korea,Czech Republic,"Estadio Akron, Zapopan"
2,A,25,2026-06-18,12:00 p.m. UTC-4,Czech Republic,South Africa,"Mercedes-Benz Stadium, Atlanta"
3,A,28,2026-06-18,7:00 p.m. UTC-6,Mexico,South Korea,"Estadio Akron, Zapopan"
4,A,53,2026-06-24,7:00 p.m. UTC-6,Czech Republic,Mexico,"Estadio Azteca, Mexico City"


## Step 2 - Handle missing team features

Three fixture teams currently have no player-feature rows: Algeria, Saudi Arabia, and Iraq.

Instead of crashing, the pipeline adds low-confidence placeholder rows. This is not ideal, but it is better than deleting fixtures. The model can still run, and the `selection_method` marks these rows as `missing_data_placeholder`.

In [3]:
fixture_codes = set(fixtures['team_a'].map(NAME_TO_CODE).dropna()) | set(fixtures['team_b'].map(NAME_TO_CODE).dropna())
teams = add_missing_team_placeholders(teams, fixture_codes)

print('Teams after placeholders:', teams.shape)
display(teams[teams['selection_method'].eq('missing_data_placeholder')])

Added missing-data placeholders for: ['DZA', 'IRQ', 'SAU']
Teams after placeholders: (48, 26)


,nation_code,nation_name,selection_method,squad_players_used,estimated_players_used,verified_supplemental_used,avg_selection_score,top6_goal_form_sum,top6_goal_form_mean,top10_chance_form_sum,...,attack_score,creation_score,defense_score,keeper_score,depth_score,data_confidence_score,coverage_pct,has_announced_squad_file,raw_power_score,power_score
45,DZA,Algeria,missing_data_placeholder,0,0,0,NaN,NaN,NaN,NaN,...,26.688889,27.977778,29.111111,18.888889,21.777778,20.0,0.0,False,NaN,29.475504
46,IRQ,Iraq,missing_data_placeholder,0,0,0,NaN,NaN,NaN,NaN,...,26.688889,27.977778,29.111111,18.888889,21.777778,20.0,0.0,False,NaN,29.475504
47,SAU,Saudi Arabia,missing_data_placeholder,0,0,0,NaN,NaN,NaN,NaN,...,26.688889,27.977778,29.111111,18.888889,21.777778,20.0,0.0,False,NaN,29.475504


## Step 3 - Clean historical results

The raw `results.csv` has nearly 50,000 matches, but we do not want to train on 1870s football. The modern game is different.

For this first model, we use matches from 2018 onward. That gives recent international football while keeping enough rows to train on.

The code also handles mixed date formats. Your file contains both `1872-11-30` and `24-06-2026` style dates.

In [4]:
recent_results = prepare_results(results, min_year=2018)

print('recent_results:', recent_results.shape)
print(recent_results['result'].value_counts())
display(recent_results.head())

recent_results: (1038, 12)
result
home_win    463
away_win    296
draw        279
Name: count, dtype: int64


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_code,away_code,result
41272,2018-01-28,United States,Bosnia and Herzegovina,0.0,0.0,Friendly,Carson,United States,False,USA,BIH,draw
41276,2018-01-31,Mexico,Bosnia and Herzegovina,1.0,0.0,Friendly,San Antonio,United States,True,MEX,BIH,home_win
41279,2018-02-28,Iraq,Saudi Arabia,4.0,1.0,Friendly,Basra,Iraq,False,IRQ,SAU,home_win
41283,2018-03-21,Iraq,Qatar,2.0,3.0,Friendly,Basra,Iraq,False,IRQ,QAT,away_win
41312,2018-03-23,France,Colombia,2.0,3.0,Friendly,Saint-Denis,France,False,FRA,COL,away_win


## Step 4 - Build training rows

A single training row is one match.

For each historical match, we create features like:

- `power_score_diff`: home team power minus away team power
- `attack_score_diff`
- `defense_score_diff`
- `keeper_score_diff`
- `attack_vs_defense_diff`

This is a common sports-analytics trick: models usually learn better from differences between opponents than from two separate raw team rows.

In [5]:
X, y = build_training_data(recent_results, teams)

print('X:', X.shape)
print('y:', y.shape)
print(y.value_counts())
display(X.head())

X: (974, 16)
y: (974,)
result
home_win    433
away_win    281
draw        260
Name: count, dtype: int64


,power_score_diff,power_score_absdiff,attack_score_diff,attack_score_absdiff,creation_score_diff,creation_score_absdiff,defense_score_diff,defense_score_absdiff,keeper_score_diff,keeper_score_absdiff,depth_score_diff,depth_score_absdiff,data_confidence_score_diff,data_confidence_score_absdiff,attack_vs_defense_diff,defense_vs_attack_diff
0,6.416253,6.416253,14.888889,14.888889,26.222222,26.222222,-25.333333,25.333333,-37.777778,37.777778,2.222222,2.222222,28.269231,28.269231,-6.000000,-4.444444
1,0.829417,0.829417,-4.333333,4.333333,6.833333,6.833333,14.888889,14.888889,-37.777778,37.777778,31.111111,31.111111,-4.038462,4.038462,-25.222222,35.777778
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-2.422222,2.422222
3,-3.879919,3.879919,-4.422222,4.422222,-12.744444,12.744444,-12.222222,12.222222,0.000000,0.000000,-36.000000,36.000000,-45.000000,45.000000,-14.644444,-2.000000
4,28.262784,28.262784,39.222222,39.222222,8.444444,8.444444,8.444444,8.444444,36.666667,36.666667,40.000000,40.000000,17.500000,17.500000,59.444444,-11.777778


## Step 5 - Train and evaluate models

We train two simple models:

- Logistic Regression: simple, explainable baseline
- Random Forest: non-linear model that can learn interactions

Football is noisy, so do not expect 80-90% accuracy. Draws especially are hard to predict. If a model gets around 45-55% three-class accuracy, that is already a plausible early baseline.

In [6]:
logistic, forest, metrics = train_models(X, y)
display(metrics)

best_model_name = metrics.sort_values('accuracy', ascending=False).iloc[0]['model']
best_model = logistic if best_model_name == 'logistic_regression' else forest
print('Best validation model:', best_model_name)


logistic_regression
              precision    recall  f1-score   support

    away_win       0.41      0.36      0.38        70
        draw       0.33      0.06      0.10        65
    home_win       0.51      0.81      0.63       109

    accuracy                           0.48       244
   macro avg       0.42      0.41      0.37       244
weighted avg       0.44      0.48      0.42       244


random_forest
              precision    recall  f1-score   support

    away_win       0.40      0.51      0.45        70
        draw       0.29      0.25      0.27        65
    home_win       0.58      0.52      0.55       109

    accuracy                           0.45       244
   macro avg       0.42      0.43      0.42       244
weighted avg       0.45      0.45      0.44       244



,model,accuracy
0,logistic_regression,0.479508
1,random_forest,0.446721


Best validation model: logistic_regression


## Step 6 - Predict 2026 group-stage fixtures

Now we apply the best validation model to the 72 group-stage fixtures.

For each fixture, the model outputs:

- Team A win probability
- draw probability
- Team B win probability
- most likely result class

Remember: these are not final betting odds. They are model outputs based on our current features.

In [7]:
predictions = predict_fixtures(fixtures, teams, best_model)

print('predictions:', predictions.shape)
display(predictions.head(20))

predictions: (72, 14)


,group,match_number,date,time_utc_offset,team_a,team_b,venue,team_a_code,team_b_code,prediction_status,team_a_win_prob,draw_prob,team_b_win_prob,predicted_result
0,A,1,2026-06-11,1:00 p.m. UTC-6,Mexico,South Africa,"Estadio Azteca, Mexico City",MEX,RSA,ok,0.236510,0.109930,0.653559,away_win
1,A,2,2026-06-11,8:00 p.m. UTC-6,South Korea,Czech Republic,"Estadio Akron, Zapopan",KOR,CZE,ok,0.488712,0.247112,0.264176,home_win
2,A,25,2026-06-18,12:00 p.m. UTC-4,Czech Republic,South Africa,"Mercedes-Benz Stadium, Atlanta",CZE,RSA,ok,0.303691,0.142634,0.553675,away_win
3,A,28,2026-06-18,7:00 p.m. UTC-6,Mexico,South Korea,"Estadio Akron, Zapopan",MEX,KOR,ok,0.372346,0.337260,0.290394,home_win
4,A,53,2026-06-24,7:00 p.m. UTC-6,Czech Republic,Mexico,"Estadio Azteca, Mexico City",CZE,MEX,ok,0.444450,0.290576,0.264974,home_win
5,A,54,2026-06-24,7:00 p.m. UTC-6,South Africa,South Korea,"Estadio BBVA, Guadalupe",RSA,KOR,ok,0.375879,0.376516,0.247605,draw
6,B,3,2026-06-12,3:00 p.m. UTC-4,Canada,Bosnia and Herzegovina,"BMO Field, Toronto",CAN,BIH,ok,0.354799,0.298528,0.346672,home_win
7,B,8,2026-06-13,12:00 p.m. UTC-7,Qatar,Switzerland,"Levi's Stadium, Santa Clara",QAT,SUI,ok,0.174070,0.324784,0.501147,away_win
8,B,26,2026-06-18,12:00 p.m. UTC-7,Switzerland,Bosnia and Herzegovina,"SoFi Stadium, Inglewood",SUI,BIH,ok,0.593602,0.218594,0.187804,home_win
9,B,27,2026-06-18,3:00 p.m. UTC-7,Canada,Qatar,"BC Place, Vancouver",CAN,QAT,ok,0.375076,0.270950,0.353974,home_win


## Step 7 - Build expected group tables

Instead of hard-assigning every match as win/draw/loss, we use expected points:

- win probability contributes `3 * P(win)`
- draw probability contributes `1 * P(draw)`

This is more stable than saying a 36% win probability definitely means a win.

In [8]:
group_tables = build_group_tables(predictions)
display(group_tables)

,group,team,points,gf_x,ga_x,gd_x
1,A,South Africa,5.378420,3.795307,2.204693,1.590614
2,A,South Korea,4.041023,3.014311,2.985689,0.028622
3,A,Czech Republic,3.717273,2.704956,3.295044,-0.590089
0,A,Mexico,3.359256,2.485426,3.514574,-1.029147
7,B,Switzerland,5.845185,4.150017,1.849983,2.300033
5,B,Bosnia and Herzegovina,3.587144,2.641560,3.358440,-0.716880
4,B,Canada,3.525242,2.612088,3.387912,-0.775824
6,B,Qatar,3.480001,2.596335,3.403665,-0.807330
8,C,Brazil,5.682185,4.082441,1.917559,2.164882
9,C,Morocco,4.588535,3.380596,2.619404,0.761193


## Step 8 - Save outputs

These outputs can later feed a frontend or a tournament simulator.

In [10]:
metrics_path = PROCESSED_DIR / 'trained_model_metrics.csv'
predictions_path = PROCESSED_DIR / 'wc_2026_group_predictions.csv'
tables_path = PROCESSED_DIR / 'wc_2026_predicted_group_tables.csv'

metrics.to_csv(metrics_path, index=False)
predictions.to_csv(predictions_path, index=False)
group_tables.to_csv(tables_path, index=False)

print('Saved:', metrics_path)
print('Saved:', predictions_path)
print('Saved:', tables_path)

Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\trained_model_metrics.csv
Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\wc_2026_group_predictions.csv
Saved: C:\Users\sambi\OneDrive\Desktop\worldcup-predictor\data\processed\wc_2026_predicted_group_tables.csv


## What to improve next

The biggest weakness now is historical feature mismatch. We trained on recent historical matches using 2026 squad features. Better future features:

- FIFA ranking at match date
- Elo rating at match date
- confederation strength
- home/neutral/away flag
- tournament type
- penalty for missing squad/player data

After that, we can build a Monte Carlo tournament simulator.